# Exercise 4.2 — Probability, odds and log-odds (with solutions)

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day4/day4-02-logistic-regression.html>.*

*Open it in Jupyter or VS Code and run it a cell at a time. Keep it in the folder that holds your `data` folder. This version has the worked answers filled in.*

In Exercise 4.1, `predict()` gave each person a **probability** of an event.
This exercise shows where it comes from. A probability must stay between 0 and
1, and a straight line does not. Logistic regression fits the straight line on
a different scale, the **log-odds**, and converts back.

> **The plots are for looking at**
>
> The code for every plot is included, so that you can run it and see the figure. You are not expected to write it yourself.

### Packages and data

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression

The same 750 people and the same split as Exercise 4.1: every fifth person is
held back as the test set.

In [ ]:
cvd = pd.read_csv("data/cvd_risk.csv")
cvd["cvd"] = (cvd["cvd"] == "yes").astype(int)       # 1 = yes, 0 = no

is_test = cvd.index % 5 == 0               # every fifth person is held back
train = cvd[~is_test]
test  = cvd[is_test]

------------------------------------------------------------------------

### Part 1 — Probability, odds and log-odds

Before fitting any model we need to be clear about how three ways of describing
the same chance are related:

- **Probability** $p$: the chance of the event happening, between 0 and 1.
- **Odds**: the probability of the event divided by the probability of no event,
  $$\text{odds} = \frac{p}{1-p}$$
  between 0 and infinity.
- **Log-odds**, also called the **logit**: the natural log of the odds,
  $$\text{logit}(p) = \log\left(\frac{p}{1-p}\right)$$
  which can be any number, negative or positive.

Start from a probability of 0.8:

In [ ]:
prob     = 0.8
odds     = prob / (1 - prob)
log_odds = np.log(odds)

print("probability:", prob)
print("odds:       ", round(odds, 3))
print("log-odds:   ", round(log_odds, 3))

A probability of 0.8 means odds of 4 ("four to one"), and log-odds of 1.386.

**Complete this table** on paper, using a calculator for the logs. The first row
is done for you.

| Probability $p$ | Odds $p/(1-p)$ | Log-odds $\log(\text{odds})$ |
|:-:|:-:|:-:|
| 0.8 | 4 | 1.386 |
| 0.5 | | |
| 0.2 | | |
| 0.9 | | |

Then check it in code:

> **Worked answer**

In [ ]:
table = pd.DataFrame({"prob": [0.8, 0.5, 0.2, 0.9]})
table["odds"]     = table["prob"] / (1 - table["prob"])
table["log_odds"] = np.log(table["odds"])

table.round(3)

> | Probability $p$ | Odds | Log-odds |
> |:-:|:-:|:-:|
> | 0.8 | 4 | 1.386 |
> | 0.5 | 1 | 0 |
> | 0.2 | 0.25 | −1.386 |
> | 0.9 | 9 | 2.197 |
>
> A probability of 0.5 is odds of 1 ("evens") and log-odds of 0. Probabilities
> below 0.5 have negative log-odds, and 0.2 and 0.8 sit at the same distance from
> 0 on either side.

Going back the other way, from log-odds $z$ to probability, is the **logistic**
(or **sigmoid**) function:

$$p = \frac{1}{1 + e^{-z}}$$

It takes any number and squeezes it into the range 0 to 1:

In [ ]:
log_odds = np.linspace(-6, 6, 121)
probability = 1 / (1 + np.exp(-log_odds))

fig, ax = plt.subplots()
ax.plot(log_odds, probability, color="red", linewidth=2)
ax.axhline(0, color="grey", linestyle="--")
ax.axhline(1, color="grey", linestyle="--")
ax.set_xlabel("Log-odds z")
ax.set_ylabel("Probability p = 1 / (1 + exp(-z))")
plt.show()

> **Learning points**
>
> - Probability lies between 0 and 1, odds between 0 and infinity, log-odds anywhere.
> - A probability of 0.5 is odds of 1 and log-odds of 0.
> - Logistic regression is a straight line on the log-odds scale; the sigmoid turns it back into a probability.

------------------------------------------------------------------------

### Part 2 — The model on the log-odds scale

Fit the model from Exercise 4.1, Part 1: an event from age only. The model
predicts the probability of 1 (an event). `C=np.inf` switches off scikit-learn's default penalty, so the coefficients
match R's.

In [ ]:
logistic_fit_age = LogisticRegression(C=np.inf)
logistic_fit_age = logistic_fit_age.fit(train[["age"]], train["cvd"])

intercept = logistic_fit_age.intercept_[0]
slope     = logistic_fit_age.coef_[0, 0]

print("intercept:", round(intercept, 4))
print("slope:    ", round(slope, 4))

This corresponds to the model

$$\text{logit}\big(P(\text{event} = \text{yes} \mid \text{age})\big)
= \beta_0 + \beta_1 \times \text{age}$$

and with the fitted coefficients

$$\text{log-odds of an event} = -5.638 + 0.0832 \times \text{age}$$

To get the probability for a given age, apply the sigmoid:

$$\hat{p} = \frac{1}{1 + \exp\big(-(\beta_0 + \beta_1 \times \text{age})\big)}$$

**Reading the slope.** Each extra year of age adds 0.0832 to the log-odds of an
event. Log-odds are hard to picture, so turn it into odds: adding 0.0832 to the
log-odds multiplies the odds by $e^{0.0832} = 1.087$. Ten extra years multiply
the odds by $e^{10 \times 0.0832} = 2.30$: they more than double. This
number, $e^{\beta_1}$, is the **odds ratio**.

> **Learning points**
>
> - Logistic regression is linear in the log-odds: $\text{log-odds} = \beta_0 + \beta_1 x$.
> - The slope is the change in log-odds for one unit of the predictor; $e^{\beta_1}$ is the odds ratio.
> - In R the model is about the second level of the outcome factor. Check the levels before you read the signs.

------------------------------------------------------------------------

### Part 3 — From the model to odds and probability

What does the model say about a 50-year-old? **By hand
first**, using the rounded coefficients:

- log-odds: $-5.638 + 0.0832 \times 50 = -1.478$
- odds: $e^{-1.478} = 0.228$
- probability: $\dfrac{0.228}{1 + 0.228} = 0.186$

Now in code, with the coefficients unrounded:

In [ ]:
age_value = 50

log_odds_50 = intercept + slope * age_value   # the straight line
odds_50     = np.exp(log_odds_50)                # odds
prob_50     = 1 / (1 + np.exp(-log_odds_50))     # probability

print("log-odds:   ", round(log_odds_50, 3))
print("odds:       ", round(odds_50, 3))
print("probability:", round(prob_50, 3))

This is the same as asking the model, with `.predict_proba()`:

In [ ]:
logistic_fit_age.predict_proba(pd.DataFrame({"age": [50]}))

A 50-year-old has a predicted probability of an event of 0.186: odds of 0.23,
or roughly one to four. The first column is the
probability of 0 (no event), the second the probability of 1.

**Your turn.** Do the same for a 70-year-old: first by hand, then by
changing the code.

> **Worked answer**

In [ ]:
age_value = 70

log_odds_70 = intercept + slope * age_value
odds_70     = np.exp(log_odds_70)
prob_70     = 1 / (1 + np.exp(-log_odds_70))

print("log-odds:   ", round(log_odds_70, 3))
print("odds:       ", round(odds_70, 3))
print("probability:", round(prob_70, 3))

> By hand: $-5.638 + 0.0832 \times 70 = 0.186$, odds $e^{0.186} = 1.20$,
> probability $1.20 / 2.20 = 0.55$. In code: log-odds 0.187, odds 1.206,
> probability 0.547. The 20 extra years multiplied the odds by
> $1.206 / 0.228 = 5.29$, which is $2.30^2$: two steps of ten years.

**This is the 70-year-old from Exercise 4.1.** The log-odds, 0.19, are just
above 0, so the odds are just above 1 and the probability just above 0.5
(0.547). That is why the model only just said `yes`.

> **Learning points**
>
> - From the fitted line to a probability: compute the log-odds, exponentiate to get the odds, then $p = \text{odds} / (1 + \text{odds})$.
> - `predict(type = "prob")` and `.predict_proba()` do exactly this calculation for you.
> - A log-odds of 0 is a probability of 0.5: that is where the predicted class changes, which Exercise 4.4 builds on.

## Stretch

> **Stretch — add blood pressure**
>
> Fit `cvd ~ age + systolic_bp` on the same training rows. Does the coefficient
> for age change? What does it now mean, in words?